# 01 — The model and the joint score

Companion to **sections 2–6 of `main.pdf`** (Gaussian AR(1) + OU diffusion).

A sequence of $K$ frames follows a stationary Gaussian AR(1) chain
$a_{k+1}=\alpha a_k+\eta_k$, $\eta_k\sim\mathcal N(0,1-\alpha^2)$, so that
$\mathrm{Var}(a_k)=1$ for every $k$.  Each frame is independently corrupted by
the variance-preserving OU channel
$x_k = e^{-t}a_k+\xi_k$, $\xi_k\sim\mathcal N(0,\Delta_t)$, $\Delta_t=1-e^{-2t}$.

We verify, numerically and at machine precision:
1. the clean covariance $\Sigma_0[i,j]=\alpha^{|i-j|}$ and its **tridiagonal** inverse $Q_0$;
2. the noised law $\Sigma_t=e^{-2t}\Sigma_0+\Delta_t I$;
3. the joint score **two ways** — direct $S=-\Sigma_t^{-1}x$ vs Tweedie
   $S_k=(e^{-t}\langle a_k\rangle_{a|x}-x_k)/\Delta_t$ — and that they agree;
4. the fully explicit $K=2$ case, where the inter-frame coupling is the single
   number $r=\alpha e^{-2t}$.

*Rule of the project: nothing is claimed without a numerical check. The full
gate is `code/numerical_audit.py` (72/72 passing); this notebook re-derives
the model-level facts interactively.*

In [1]:
import sys
sys.path.insert(0, 'code')
import numpy as np
np.set_printoptions(precision=6, suppress=True)
from ar1_utils import (ar1_covariance, ar1_precision_clean, ou_params, sigma_t,
                       precision_t, joint_score_matrix, joint_score_via_tweedie,
                       gaussian_posterior)
print('model code loaded')

model code loaded


## 1. Clean joint law: dense covariance, tridiagonal precision

The covariance is dense Toeplitz ($\alpha^{|i-j|}$: correlations never vanish),
but its inverse is **exactly tridiagonal** — the algebraic shadow of the Markov
property.  Interior diagonal $(1+\alpha^2)/\sigma_\eta^2$, boundaries
$1/\sigma_\eta^2$, off-diagonal $-\alpha/\sigma_\eta^2$
(`main.pdf` Prop. 5).

In [2]:
K, alpha = 6, 0.8
Sigma0 = ar1_covariance(K, alpha)
Q0 = ar1_precision_clean(K, alpha)            # the closed-form tridiagonal
print('Sigma_0 (dense Toeplitz):\n', Sigma0)
print('\nQ0 from the closed form:\n', Q0)
print('\nmax |Q0 - inv(Sigma_0)|      =', np.max(np.abs(Q0 - np.linalg.inv(Sigma0))))
mask = np.abs(np.subtract.outer(np.arange(K), np.arange(K))) > 1
print('max |off-tridiagonal entries| =', np.max(np.abs(Q0[mask])), ' (exactly zero)')

Sigma_0 (dense Toeplitz):
 [[1.      0.8     0.64    0.512   0.4096  0.32768]
 [0.8     1.      0.8     0.64    0.512   0.4096 ]
 [0.64    0.8     1.      0.8     0.64    0.512  ]
 [0.512   0.64    0.8     1.      0.8     0.64   ]
 [0.4096  0.512   0.64    0.8     1.      0.8    ]
 [0.32768 0.4096  0.512   0.64    0.8     1.     ]]

Q0 from the closed form:
 [[ 2.777778 -2.222222  0.        0.        0.        0.      ]
 [-2.222222  4.555556 -2.222222  0.        0.        0.      ]
 [ 0.       -2.222222  4.555556 -2.222222  0.        0.      ]
 [ 0.        0.       -2.222222  4.555556 -2.222222  0.      ]
 [ 0.        0.        0.       -2.222222  4.555556 -2.222222]
 [ 0.        0.        0.        0.       -2.222222  2.777778]]

max |Q0 - inv(Sigma_0)|      = 1.7763568394002505e-15
max |off-tridiagonal entries| = 0.0  (exactly zero)


## 2. Noised joint law

$x=e^{-t}a+\xi$ with independent noise, so
$\Sigma_t=e^{-2t}\Sigma_0+\Delta_t I$.  The diagonal stays $1$ for all $t$
(variance preservation): the channel trades signal for noise at constant total
variance.

In [3]:
t = 0.4
St = sigma_t(Sigma0, t)
print('Sigma_t diagonal (should be all 1):', np.diag(St))
print('Sigma_t[0,1] =', St[0,1], ' = alpha e^{-2t} =', alpha*np.exp(-2*t))
Qt = precision_t(Sigma0, t)
print('\nmax |Sigma_t @ Q_t - I| =', np.max(np.abs(St @ Qt - np.eye(K))))

Sigma_t diagonal (should be all 1): [1. 1. 1. 1. 1. 1.]
Sigma_t[0,1] = 0.35946317129377725  = alpha e^{-2t} = 0.35946317129377725

max |Sigma_t @ Q_t - I| = 2.220446049250313e-16


## 3. The score, two ways

**Direct**: $S(x,t)=-\Sigma_t^{-1}x$ (one matrix inversion).

**Bayesian (Tweedie, `main.pdf` Th. 7)**:
$S_k=(e^{-t}\,\mathbb E[a_k\,|\,x]-x_k)/\Delta_t$ — compute the posterior mean
of the clean frame given the **whole** noisy sequence, then rescale.

Two completely different computational routes; they must give the same vector.
This is a real end-to-end check of every formula above.

In [4]:
rng = np.random.default_rng(0)
x = rng.standard_normal(K)
S_direct  = joint_score_matrix(x, t, Sigma0, alpha)
S_tweedie = joint_score_via_tweedie(x, t, Sigma0, alpha)
print('S (direct)  =', S_direct)
print('S (Tweedie) =', S_tweedie)
print('max abs difference =', np.max(np.abs(S_direct - S_tweedie)))

S (direct)  = [-0.08968   0.380272 -0.874232 -0.059928  0.929661 -0.534201]
S (Tweedie) = [-0.08968   0.380272 -0.874232 -0.059928  0.929661 -0.534201]
max abs difference = 2.220446049250313e-16


## 4. $K=2$: everything explicit

With unit variances ('main.pdf' section 6):

$$\Sigma_t=\begin{pmatrix}1&r\\ r&1\end{pmatrix},\qquad
S_0=-\frac{x_0-r\,x_1}{1-r^2},\qquad r=\alpha e^{-2t}.$$

The cross term $-r\,x_1$ in $S_0$ is the **inter-frame coupling**: the denoiser
of frame 0 borrows evidence from frame 1.  The per-frame *marginal* score is
just $-x_0$ at every $t$ (each $x_k$ is $\mathcal N(0,1)$ by variance
preservation) — it knows nothing about the neighbour.  This tiny example is
why the *joint* score is the right object (Mézard's correction, `main.pdf`
Remark 1).

In [5]:
from chain_formulas import k2_sigma_t, k2_precision_t, k2_score

alpha2 = 0.7
for t2 in (0.05, 0.5, 2.0):
    r = alpha2*np.exp(-2*t2)
    x2 = np.array([1.0, -0.5])
    S_closed  = k2_score(x2, alpha2, t2)
    S_generic = joint_score_matrix(x2, t2, ar1_covariance(2, alpha2), alpha2)
    print(f't={t2:<4}  r=alpha e^-2t={r:.4f}   S_closed={S_closed}'
          f'   max|closed-generic|={np.max(np.abs(S_closed-S_generic)):.2e}')
print('\nmarginal score of x_0 would be -x_0 = -1.0 at EVERY t: '
      'all the t-dependence above is joint structure.')

t=0.05  r=alpha e^-2t=0.6334   S_closed=[-2.198806  1.892693]   max|closed-generic|=2.22e-16
t=0.5   r=alpha e^-2t=0.2575   S_closed=[-1.208927  0.811318]   max|closed-generic|=0.00e+00
t=2.0   r=alpha e^-2t=0.0128   S_closed=[-1.006576  0.512905]   max|closed-generic|=0.00e+00

marginal score of x_0 would be -x_0 = -1.0 at EVERY t: all the t-dependence above is joint structure.


## 5. The posterior behind Tweedie

$p(a|x)=\mathcal N(m, J^{-1})$ with $J=(e^{-2t}/\Delta_t)I+Q_0$ (tridiagonal!)
and $m=J^{-1}h$, $h=(e^{-t}/\Delta_t)x$ (`main.pdf` Prop. 9).  We verify the
posterior mean is what Tweedie needs, and that the posterior precision is
exactly 'evidence + prior'.

In [6]:
mu, Delta = ou_params(t)
mu_post, Sig_post = gaussian_posterior(x, t, Sigma0, alpha)
J = (mu*mu/Delta)*np.eye(K) + Q0
h = (mu/Delta)*x
print('max |J^-1 h - posterior mean| =', np.max(np.abs(np.linalg.solve(J, h) - mu_post)))
print('max |inv(J) - posterior cov | =', np.max(np.abs(np.linalg.inv(J) - Sig_post)))
S_from_post = (mu*mu_post - x)/Delta
print('max |Tweedie(posterior mean) - S_direct| =', np.max(np.abs(S_from_post - S_direct)))

max |J^-1 h - posterior mean| = 6.938893903907228e-17
max |inv(J) - posterior cov | = 1.1102230246251565e-16
max |Tweedie(posterior mean) - S_direct| = 2.220446049250313e-16


## Summary

* $Q_0$ tridiagonal at machine precision — Markov structure made algebra.
* $\Sigma_t = e^{-2t}\Sigma_0+\Delta_t I$, unit diagonal for all $t$.
* Direct score $=$ Tweedie score to $\sim10^{-15}$.
* $K=2$ closed form: coupling $r=\alpha e^{-2t}$, invisible to the marginal score.
* Posterior: $J = \text{evidence}\cdot I + Q_0$, mean linear in $x$.

Continue with **`02_belief_propagation.ipynb`** for the factor graph and the
message-passing computation of the same score in $O(K)$.